# NinaPro 原始 sEMG 泊松脉冲编码

本 Notebook 直接读取 `ninapro_data/processed/exerciseA/slide_window` 中未经归一化的原始 sEMG，将同一个 200 ms 窗口编码为 `T=40、80、120` 的二值泊松脉冲。编码结果不会再进行 Z-score、Min-Max 或其他归一化。

生成两种表示：

1. `offset_128`：原始值加 128 后映射到 0～100 Hz，输出 `[B, 16, T]`；
2. `polarity_split`：正负极性分离，固定幅值参考为 64，使用 0.57 次幂增强中低幅值，再映射到 0～200 Hz，输出 `[B, 32, T]`。

目标时间步始终覆盖相同的 200 ms。原始 40 点序列通过零阶保持映射到目标 `T`，每个时间箱以 `p = 1 - exp(-rate * dt)` 独立采样。固定随机种子保证相同参数下结果可复现。


In [1]:
from datetime import datetime, timezone
import json
from pathlib import Path

import numpy as np


def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        source = candidate / 'ninapro_data' / 'processed' / 'exerciseA' / 'slide_window'
        if (source / 'train.npz').is_file() and (source / 'test.npz').is_file():
            return candidate.resolve()
    raise FileNotFoundError('无法定位包含 NinaPro 滑窗数据的项目根目录')


PROJECT_ROOT = find_project_root()
SOURCE_DIR = PROJECT_ROOT / 'ninapro_data' / 'processed' / 'exerciseA' / 'slide_window'
OUTPUT_ROOT = PROJECT_ROOT / 'ninapro_data' / 'raw_data_poisson_spikes'

print(f'项目根目录：{PROJECT_ROOT}')
print(f'输入目录：{SOURCE_DIR}')
print(f'输出目录：{OUTPUT_ROOT}')


项目根目录：C:\Users\Fortyfour\Desktop\SNN_for_Ninapro
输入目录：C:\Users\Fortyfour\Desktop\SNN_for_Ninapro\ninapro_data\processed\exerciseA\slide_window
输出目录：C:\Users\Fortyfour\Desktop\SNN_for_Ninapro\ninapro_data\raw_data_poisson_spikes


In [2]:
# 所有 T 都表示同一个 200 ms 物理窗口，只改变离散时间箱宽度。
T_VALUES = (40, 80, 120)
WINDOW_SECONDS = 0.2
SOURCE_CHANNELS = 16
SOURCE_TIME_STEPS = 40
RANDOM_SEED = 42
CHUNK_SIZE = 512
OVERWRITE = True

ENCODINGS = {
    'offset_128': {
        'description': 'raw + 128, fixed full-scale linear rate mapping',
        'output_channels': 16,
        'max_rate_hz': 100.0,
        'raw_shift': 128.0,
        'amplitude_full_scale': 255.0,
        'channel_order': '[ch0, ..., ch15]',
    },
    'polarity_split': {
        'description': 'positive/negative polarity split with compressive amplitude mapping',
        'output_channels': 32,
        'max_rate_hz': 200.0,
        'amplitude_reference': 64.0,
        'amplitude_exponent': 0.57,
        'channel_order': '[ch0+, ..., ch15+, ch0-, ..., ch15-]',
    },
}

ENCODING_IDS = {'offset_128': 1, 'polarity_split': 2}
SPLIT_IDS = {'train': 1, 'test': 2}

print(json.dumps(ENCODINGS, ensure_ascii=False, indent=2))


{
  "offset_128": {
    "description": "raw + 128, fixed full-scale linear rate mapping",
    "output_channels": 16,
    "max_rate_hz": 100.0,
    "raw_shift": 128.0,
    "amplitude_full_scale": 255.0,
    "channel_order": "[ch0, ..., ch15]"
  },
  "polarity_split": {
    "description": "positive/negative polarity split with compressive amplitude mapping",
    "output_channels": 32,
    "max_rate_hz": 200.0,
    "amplitude_reference": 64.0,
    "amplitude_exponent": 0.57,
    "channel_order": "[ch0+, ..., ch15+, ch0-, ..., ch15-]"
  }
}


In [3]:
def load_source_split(split_name):
    source_path = SOURCE_DIR / f'{split_name}.npz'
    with np.load(source_path, allow_pickle=False) as data:
        missing = {'X', 'y'}.difference(data.files)
        if missing:
            raise KeyError(f'{source_path} 缺少字段：{sorted(missing)}')
        features = np.asarray(data['X'], dtype=np.float32).copy()
        labels = np.asarray(data['y'], dtype=np.int64).copy()

    expected_tail = (SOURCE_CHANNELS, SOURCE_TIME_STEPS)
    if features.ndim != 3 or features.shape[1:] != expected_tail:
        raise ValueError(f'{split_name} X 形状错误：{features.shape}')
    if labels.shape != (features.shape[0],):
        raise ValueError(f'{split_name} y 形状错误：{labels.shape}')
    if not np.isfinite(features).all():
        raise ValueError(f'{split_name} X 包含 NaN 或无穷值')
    return features, labels


def target_source_indices(target_steps):
    # 零阶保持保证任意目标 T 都覆盖同一个 200 ms，而不是改变窗口时长。
    indices = np.floor(
        np.arange(target_steps, dtype=np.float64)
        * SOURCE_TIME_STEPS
        / target_steps
    ).astype(np.int64)
    return np.clip(indices, 0, SOURCE_TIME_STEPS - 1)


def make_rng(encoding_name, target_steps, split_name):
    # 不使用 Python hash，避免不同进程的哈希随机化破坏复现性。
    seed = np.random.SeedSequence(
        [
            RANDOM_SEED,
            ENCODING_IDS[encoding_name],
            int(target_steps),
            SPLIT_IDS[split_name],
        ]
    )
    return np.random.default_rng(seed)


def offset_128_rates(x):
    config = ENCODINGS['offset_128']
    shifted = np.clip(x + config['raw_shift'], 0.0, 255.0)
    return shifted * (config['max_rate_hz'] / config['amplitude_full_scale'])


def polarity_split_rates(x):
    config = ENCODINGS['polarity_split']
    positive = np.maximum(x, 0.0)
    negative = np.maximum(-x, 0.0)
    amplitude = np.concatenate([positive, negative], axis=1)
    amplitude = np.minimum(amplitude / config['amplitude_reference'], 1.0)

    # 压缩映射提升常见中低幅值的脉冲密度，同时限制极端值的最大率。
    amplitude = np.power(amplitude, config['amplitude_exponent'])
    return amplitude * config['max_rate_hz']


RATE_FUNCTIONS = {
    'offset_128': offset_128_rates,
    'polarity_split': polarity_split_rates,
}


In [4]:
def encode_features(features, encoding_name, target_steps, rng):
    config = ENCODINGS[encoding_name]
    output_shape = (
        features.shape[0],
        config['output_channels'],
        target_steps,
    )
    encoded = np.empty(output_shape, dtype=np.uint8)
    source_indices = target_source_indices(target_steps)
    bin_seconds = WINDOW_SECONDS / target_steps
    expected_spike_sum = 0.0

    for start in range(0, features.shape[0], CHUNK_SIZE):
        end = min(start + CHUNK_SIZE, features.shape[0])
        held = features[start:end][:, :, source_indices]
        rates = RATE_FUNCTIONS[encoding_name](held)

        # 二值时间箱记录泊松过程中是否至少出现一次事件。
        probabilities = -np.expm1(-rates * bin_seconds)
        random_values = rng.random(probabilities.shape, dtype=np.float32)
        encoded[start:end] = (random_values < probabilities).astype(np.uint8)
        expected_spike_sum += probabilities.sum(dtype=np.float64)

    sample_count = features.shape[0]
    value_count = encoded.size
    spike_count = int(encoded.sum(dtype=np.int64))
    stats = {
        'shape': list(encoded.shape),
        'dtype': str(encoded.dtype),
        'spike_count': spike_count,
        'spike_density': spike_count / value_count,
        'expected_spike_density': expected_spike_sum / value_count,
        'mean_spikes_per_window': spike_count / sample_count,
        'expected_mean_spikes_per_window': expected_spike_sum / sample_count,
    }
    return encoded, stats


def save_npz_atomic(path, features, labels):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f'{path.stem}.tmp{path.suffix}')
    if path.exists() and not OVERWRITE:
        raise FileExistsError(f'输出已存在：{path}')
    if temporary.exists():
        temporary.unlink()
    np.savez_compressed(temporary, X=features, y=labels)
    temporary.replace(path)


def write_json_atomic(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f'{path.name}.tmp')
    if path.exists() and not OVERWRITE:
        raise FileExistsError(f'输出已存在：{path}')
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )
    temporary.replace(path)


def validate_saved_split(path, labels, expected_shape):
    with np.load(path, allow_pickle=False) as data:
        if set(data.files) != {'X', 'y'}:
            raise RuntimeError(f'{path} 字段错误：{data.files}')
        encoded = data['X']
        saved_labels = data['y']

        if encoded.shape != expected_shape or encoded.dtype != np.uint8:
            raise RuntimeError(
                f'{path} X 规格错误：{encoded.shape}/{encoded.dtype}'
            )
        if not np.logical_or(encoded == 0, encoded == 1).all():
            raise RuntimeError(f'{path} X 不是二值脉冲')
        if saved_labels.dtype != np.int64 or not np.array_equal(saved_labels, labels):
            raise RuntimeError(f'{path} 标签与源数据不一致')

        return {
            'verified': True,
            'file_size_bytes': path.stat().st_size,
            'minimum': int(encoded.min()),
            'maximum': int(encoded.max()),
        }


In [5]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
generated_at = datetime.now(timezone.utc).isoformat()
summary = {
    'dataset': 'NinaPro DB5 Exercise A',
    'source_directory': str(SOURCE_DIR.relative_to(PROJECT_ROOT)).replace('\\', '/'),
    'output_directory': str(OUTPUT_ROOT.relative_to(PROJECT_ROOT)).replace('\\', '/'),
    'generated_at_utc': generated_at,
    'window_seconds': WINDOW_SECONDS,
    'source_shape_layout': '[B, 16, 40]',
    'target_time_steps': list(T_VALUES),
    'random_seed': RANDOM_SEED,
    'normalized_before_encoding': False,
    'normalized_after_encoding': False,
    'encodings': {},
}

for encoding_name, config in ENCODINGS.items():
    encoding_summary = {
        'parameters': config,
        'time_steps': {},
    }
    summary['encodings'][encoding_name] = encoding_summary

    for target_steps in T_VALUES:
        target_dir = OUTPUT_ROOT / encoding_name / f'T_{target_steps}'
        target_metadata = {
            'dataset': summary['dataset'],
            'encoding_name': encoding_name,
            'encoding_parameters': config,
            'source_directory': summary['source_directory'],
            'window_seconds': WINDOW_SECONDS,
            'target_time_steps': target_steps,
            'bin_seconds': WINDOW_SECONDS / target_steps,
            'random_seed': RANDOM_SEED,
            'normalized_before_encoding': False,
            'normalized_after_encoding': False,
            'poisson_binary_probability': '1 - exp(-rate_hz * bin_seconds)',
            'generated_at_utc': generated_at,
            'splits': {},
        }

        for split_name in ('train', 'test'):
            features, labels = load_source_split(split_name)
            rng = make_rng(encoding_name, target_steps, split_name)
            encoded, stats = encode_features(
                features,
                encoding_name,
                target_steps,
                rng,
            )
            output_path = target_dir / f'{split_name}.npz'
            expected_shape = encoded.shape
            save_npz_atomic(output_path, encoded, labels)
            del encoded

            stats.update(
                validate_saved_split(output_path, labels, expected_shape)
            )
            stats['file'] = output_path.name
            stats['label_shape'] = list(labels.shape)
            stats['label_dtype'] = str(labels.dtype)
            target_metadata['splits'][split_name] = stats
            print(
                f'{encoding_name}/T_{target_steps}/{split_name}: '
                f'X={tuple(stats["shape"])}, '
                f'density={stats["spike_density"]:.4%}, '
                f'spikes/window={stats["mean_spikes_per_window"]:.2f}'
            )
            del features, labels

        write_json_atomic(target_dir / 'metadata.json', target_metadata)
        encoding_summary['time_steps'][str(target_steps)] = target_metadata

write_json_atomic(OUTPUT_ROOT / 'metadata.json', summary)
print(f'全部编码与验证完成：{OUTPUT_ROOT}')


offset_128/T_40/train: X=(17839, 16, 40), density=22.0092%, spikes/window=140.86


offset_128/T_40/test: X=(8787, 16, 40), density=21.9886%, spikes/window=140.73


offset_128/T_80/train: X=(17839, 16, 80), density=11.7111%, spikes/window=149.90


offset_128/T_80/test: X=(8787, 16, 80), density=11.6894%, spikes/window=149.62


offset_128/T_120/train: X=(17839, 16, 120), density=7.9636%, spikes/window=152.90


...

polarity_split/T_40/test: X=(8787, 32, 40), density=10.4736%, spikes/window=134.06


polarity_split/T_80/train: X=(17839, 32, 80), density=5.6855%, spikes/window=145.55


polarity_split/T_80/test: X=(8787, 32, 80), density=5.7437%, spikes/window=147.04


polarity_split/T_120/train: X=(17839, 32, 120), density=3.9105%, spikes/window=150.16


polarity_split/T_120/test: X=(8787, 32, 120), density=3.9435%, spikes/window=151.43
全部编码与验证完成：C:\Users\Fortyfour\Desktop\SNN_for_Ninapro\ninapro_data\raw_data_poisson_spikes


In [6]:
print('\n训练集编码结果汇总')
print('encoding          T    density    spikes/window    expected/window')
for encoding_name, encoding_summary in summary['encodings'].items():
    for target_steps in T_VALUES:
        stats = encoding_summary['time_steps'][str(target_steps)]['splits']['train']
        print(
            f'{encoding_name:<17} '
            f'{target_steps:>3}  '
            f'{stats["spike_density"]:>9.4%}  '
            f'{stats["mean_spikes_per_window"]:>13.2f}  '
            f'{stats["expected_mean_spikes_per_window"]:>15.2f}'
        )



训练集编码结果汇总
encoding          T    density    spikes/window    expected/window
offset_128         40   22.0092%         140.86           140.88
offset_128         80   11.7111%         149.90           149.73
offset_128        120    7.9636%         152.90           152.86
polarity_split     40   10.3959%         133.07           133.14
polarity_split     80    5.6855%         145.55           145.62
polarity_split    120    3.9105%         150.16           150.26
